# Train the water-body segmentation U-Net (from scratch -- no prior training notebook existed)

**Goal:** given a single optical image, predict a per-pixel water/non-water mask -- the model
`models/water_segmentation/water_segmentation_tool.py` already wraps for inference.

**Why this notebook exists:** the checkpoint already in `models/water_segmentation/checkpoints/`
(`segmentation_models_pytorch` U-Net, `encoder_name="resnet34"`, 256x256 input, stored
`val_iou`=0.7638) had no training notebook or known data source on record -- it was just a
`torch.save`'d checkpoint. This notebook trains a fresh one with a known, documented pipeline, so
it's reproducible and (unlike the original) has verified preprocessing to match what
`water_segmentation_tool.py` already assumes -- ImageNet mean/std normalization, bilinear resize
to 256x256. Training with that exact convention here means the new checkpoint's BatchNorm stats
will actually be calibrated for it, resolving the "unverified preprocessing" caveat called out in
that script's docstring.

**Data:** [Satellite Images of Water Bodies](https://www.kaggle.com/datasets/franciscoescobar/satellite-images-of-water-bodies)
(Francisco Escobar) -- 2841 Sentinel-2 image/mask pairs, masks generated via NDWI thresholding.
**License: CC BY-NC-SA 4.0 -- research / non-commercial use only, and share-alike** (same
commercial-use caveat as the DIOR-RSVG grounding data -- keep this in mind before shipping the
fine-tuned weights). Add it as a notebook input before running (see the cell after Setup).

**Model:** `segmentation_models_pytorch.Unet(encoder_name="resnet34", classes=1)` -- deliberately
the same architecture as the existing checkpoint, so the exported weights are a drop-in
replacement (same `water_segmentation_tool.py`, same `checkpoint_path` argument).

Run cells top to bottom. Kaggle: enable **Internet** and a **GPU** (T4x2 or P100) in the notebook's
Settings panel before starting.

## 0. GPU compatibility check

**Found live, via an actual failed run on Kaggle**: this session's preinstalled PyTorch build
(2.10.0+cu128) only supports CUDA compute capabilities sm_70 and up -- it silently drops support
for the Pascal-generation **P100** (sm_60), one of the two GPU types Kaggle itself still offers
here (T4x2 or P100, per the note below). Landing on a P100 crashed training ~40 seconds in with
`CUDA error: no kernel image is available for execution on the device` -- a real run, not a
hypothetical. The cell below detects the actual GPU via `nvidia-smi` (no torch import needed yet,
so this runs before torch's own compute-capability list is fixed for the process) and reinstalls
a CUDA 11.8 build if the assigned GPU isn't in the preinstalled build's supported list -- CUDA 11.8
wheels cover Pascal through Hopper, so this works regardless of which GPU Kaggle happens to assign.

In [ ]:
import subprocess

try:
    cc_raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
    major, minor = cc_raw.split(".")
    needed_sm = f"sm_{major}{minor}"
except Exception as e:
    needed_sm = None
    print(f"Could not query GPU compute capability via nvidia-smi ({e}) -- skipping the compatibility check.")

if needed_sm:
    import torch as _torch_probe
    supported = _torch_probe.cuda.get_arch_list() if _torch_probe.cuda.is_available() else []
    if needed_sm not in supported:
        print(f"GPU needs {needed_sm}, not in the preinstalled torch build's supported list "
              f"{supported} -- reinstalling a CUDA 11.8 build (covers Pascal through Hopper)...")
        subprocess.run(
            ["pip", "install", "-q", "--force-reinstall",
             "torch==2.4.0", "--index-url", "https://download.pytorch.org/whl/cu118"],
            check=True,
        )
        print("Reinstalled -- the next cell's `import torch` will pick up the new build "
              "(this is the first import of torch in this process, so no stale-module issue).")
    else:
        print(f"GPU compute capability {needed_sm} already supported by the preinstalled build.")

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))


## 1. Setup

In [ ]:
!pip install -q segmentation-models-pytorch


## 2. Add the dataset

On the right-hand panel of the Kaggle notebook editor: **Add Input -> search "Satellite Images of
Water Bodies"** (author: Francisco Escobar) **-> Add**. It mounts read-only under
`/kaggle/input/satellite-images-of-water-bodies/Water Bodies Dataset/{Images,Masks}/`.

The cell below locates it by globbing for a directory that actually contains both an `Images` and
a `Masks` subfolder, rather than hardcoding the exact mount path -- Kaggle's auto-generated slug
has been known to vary slightly.


In [ ]:
import glob, os

candidates = [
    p for p in glob.glob("/kaggle/input/**/Images", recursive=True)
    if os.path.isdir(os.path.join(os.path.dirname(p), "Masks"))
]
assert candidates, (
    "No '.../Images' + '.../Masks' pair found under /kaggle/input -- did you add the "
    "'Satellite Images of Water Bodies' dataset yet? See the markdown cell above."
)
IMAGES_DIR = candidates[0]
MASKS_DIR = os.path.join(os.path.dirname(IMAGES_DIR), "Masks")
print("Images:", IMAGES_DIR)
print("Masks: ", MASKS_DIR)
print("Image count:", len(os.listdir(IMAGES_DIR)), "| Mask count:", len(os.listdir(MASKS_DIR)))


## 3. Pair images with masks and split train/val

Paired by matching filename **stem** (not assumed to be byte-identical filenames) so this doesn't
silently break if the extensions or a suffix differ between the two folders -- prints a count of
any unmatched files so a mismatch is visible rather than silently dropped.


In [ ]:
import random

def stem(filename):
    return os.path.splitext(filename)[0]

image_files = {stem(f): f for f in os.listdir(IMAGES_DIR)}
mask_files = {stem(f): f for f in os.listdir(MASKS_DIR)}

paired_stems = sorted(set(image_files) & set(mask_files))
unmatched = (set(image_files) | set(mask_files)) - set(paired_stems)
print(f"Paired: {len(paired_stems)}  |  Unmatched (dropped): {len(unmatched)}")
if unmatched:
    print("Example unmatched stems:", list(unmatched)[:5])

pairs = [(os.path.join(IMAGES_DIR, image_files[s]), os.path.join(MASKS_DIR, mask_files[s])) for s in paired_stems]

random.seed(0)
random.shuffle(pairs)
n_val = max(1, int(0.15 * len(pairs)))
val_pairs = pairs[:n_val]
train_pairs = pairs[n_val:]
print(f"train={len(train_pairs)}  val={len(val_pairs)}")


## 4. Dataset, model, loss

`IMG_SIZE=256` matches the existing checkpoint's stored `img_size` (and `water_segmentation_tool.py`'s
resize target) -- keep this unless you also update that script. Normalization matches the
ImageNet mean/std `water_segmentation_tool.py` already uses.


In [ ]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

IMG_SIZE = 256
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


class WaterBodiesDataset(Dataset):
    def __init__(self, pairs, augment=False):
        self.pairs = pairs
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        mask = Image.open(mask_path).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)

        if self.augment and random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)
        if self.augment and random.random() < 0.5:
            image = TF.vflip(image)
            mask = TF.vflip(mask)

        image_t = TF.to_tensor(image)
        image_t = TF.normalize(image_t, MEAN, STD)
        mask_t = (torch.from_numpy(np.asarray(mask, dtype=np.float32)) > 127).float().unsqueeze(0)
        return image_t, mask_t


train_ds = WaterBodiesDataset(train_pairs, augment=True)
val_ds = WaterBodiesDataset(val_pairs, augment=False)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)
print(f"train batches={len(train_loader)}  val batches={len(val_loader)}")


In [ ]:
import segmentation_models_pytorch as smp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", classes=1, activation=None).to(DEVICE)

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
print("Model on", DEVICE)


## 5. Train

`EPOCHS=20` is a reasonable starting point for ~2400 training images at this resolution on a
T4/P100 -- watch the printed val IoU and stop early (or bump it up) based on where it plateaus.
Saves the best-val-IoU weights in memory (not to disk each epoch) and writes the checkpoint once,
in Section 7, from whichever epoch had the best score.


In [ ]:
def iou_score(logits, targets, threshold=0.5, eps=1e-7):
    preds = (torch.sigmoid(logits) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = ((preds + targets) > 0).float().sum(dim=(1, 2, 3))
    return ((intersection + eps) / (union + eps)).mean().item()


EPOCHS = 20
best_val_iou = -1.0
best_state_dict = None

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_ious = []
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            val_ious.append(iou_score(logits, masks))
    val_iou = sum(val_ious) / len(val_ious)

    marker = ""
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        marker = "  <- best so far"
    print(f"epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  val_iou={val_iou:.4f}{marker}")

print(f"\nBest val IoU: {best_val_iou:.4f}  (previous checkpoint: 0.7638)")


## 6. Export

Same checkpoint dict shape as the existing one (`{"model_state_dict", "encoder_name", "img_size",
"val_iou"}`) so it's a **drop-in replacement** -- `water_segmentation_tool.py` needs no changes,
just point `checkpoint_path` at the new file (or overwrite
`models/water_segmentation/checkpoints/water_body_unet_final.pt` once you're happy with the
val IoU above vs. the old 0.7638).


In [ ]:
import os

os.makedirs("/kaggle/working/output_model", exist_ok=True)
torch.save(
    {
        "model_state_dict": best_state_dict,
        "encoder_name": "resnet34",
        "img_size": IMG_SIZE,
        "val_iou": best_val_iou,
    },
    "/kaggle/working/output_model/water_body_unet_final.pt",
)
print("Exported to /kaggle/working/output_model/water_body_unet_final.pt -- download from this notebook's Output tab.")


## Next

Download `water_body_unet_final.pt` from this notebook's Output tab. Sanity-check it locally with:

```bash
python models/water_segmentation/water_segmentation_tool.py --image <a real water-containing image> \
    --checkpoint <downloaded water_body_unet_final.pt> --draw out.jpg
```

If the val IoU above beats 0.7638 (and the overlay on a real image looks right), replace
`models/water_segmentation/checkpoints/water_body_unet_final.pt` with the new file.
